# Lab 7 solution: Serial Communication and Robot Control

Companion to `SerialCommunicationRobotControlTemplate.ipynb` (`pdfs/Serial_Communication_Robot_Control.pdf`). **The code cells are the template's own cells with the blanks filled in**
(changes beyond the blanks are commented in the cell). After each code cell:
- **Concept**: what the task computes and why;
- **How it works**: the code line by line, every new piece of syntax with a tiny example;
- sometimes an **Example**: a short check, shown as code + its real output (text only, nothing extra to run).

**Real robot by default.** Run it on the **Pi** (Jupyter there, port forwarded to the laptop). Part 1 (Tasks 6–15) talks to the robot through `/dev/serial0` with the ROS controller **off**; Part 2 (Tasks 18–25) goes through the running `mycobot_control` node. Tasks 1–5 (frames) run anywhere.
- ⚠ Run this notebook **cell by cell**, never "Run All". Cells that move the arm are marked **[MOVES]**: workspace clear,
  hand at the robot's power switch, first move small and slow.
- Cells that need the robot have **no saved output** here (they were not run on the laptop); all other outputs are real.

**Simulation (optional).** Blocks marked 🧪 show how to run the same cells **without** the robot, against
`solutions/sim_robot.py` (a simulated arm behind a virtual serial port). They come after the real-robot solution and
are never needed in the lab.

**Run from `solutions/`.** `import labpaths` (added to the setup cell) puts the repo root on `sys.path`, so
`from serial_iface import MyCobotSerialInterface` works as in the template.

**The idea of the lab.** The robot's controller understands **frames**: `FE FE LEN CMD [payload] FA`. Angles travel as signed
16-bit integers in 0.01° units. Part 1 builds and sends such frames yourself (`serial_iface.py`); Part 2 does the same
through ROS 2 topics, where the controller node owns the port and converts radians ↔ degrees.

**Safety differences from the template** (applied in the cells): every target is **relative to the measured pose**
(joint 1 ± 10°, speed 20); never all-zero angles (the arm straight up is a 99° move from the parked pose).

# Serial Communication and Robot Control for the myCobot 280 Labs
## Lab 7: Frames, Commands, State Feedback, and Safe Real-Robot Operation

Run this notebook from `~/venvs/mycobot`:

```bash
source ~/venvs/mycobot/bin/activate
jupyter lab
```

This is the first lab where you interact directly with the physical myCobot 280. You will inspect the serial protocol, send commands, read back robot state, compare commanded and measured motion, and use basic safety-stop procedures. The notebook connects low-level serial communication to the higher-level ROS 2 workflow.

**Important:** This notebook has two parts. In Part 1 (Tasks 1–15), you work with the serial interface directly — the ROS 2 controller must NOT be running. In Part 2 (Tasks 16–29), you use the ROS 2 controller — the serial object must be closed first.

In [1]:
%matplotlib widget
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import labpaths    # added: repo root on sys.path, so `import serial_iface` works from solutions/

**How it works**: `time` is used for waits (`time.sleep`) and timing (`time.time()`); `labpaths` only adjusts `sys.path`,
the list of folders Python searches for modules.

The serial interface and ROS 2 imports are loaded in their respective sections below to keep the two parts cleanly separated.

## Part 1: Direct Serial Communication

In this part you communicate directly with the myCobot 280 through the serial interface. The ROS 2 controller (`mycobot_control`) must **not** be running — it would hold the serial port exclusively.

## Setup and Codebase Inspection

### 1. Inspect the codebase files

Before running any commands, read through the following files in the `Code` folder:
- `serial_iface.py` — the low-level serial wrapper
- `mycobot_control.py` — the ROS 2 controller node (used in Part 2)

Look for:
- How the serial port is opened and configured
- How frames are assembled and parsed
- How angles are encoded (int16, degrees × 100)
- How stop/resume commands are structured
- How the ROS 2 node bridges serial commands to topics

In [2]:
# Confirm the serial interface file is available
import os
serial_iface_path = os.path.join(os.getcwd(), 'serial_iface.py')
if not os.path.isfile(serial_iface_path):        # added: the notebook runs in solutions/, the file is in the repo root
    serial_iface_path = os.path.join(labpaths.REPO_ROOT, 'serial_iface.py')
print('serial_iface.py exists:', os.path.isfile(serial_iface_path))

serial_iface.py exists: True


**Concept: a tour of `serial_iface.py`** (class `MyCobotSerialInterface`)

| piece | what it does |
|---|---|
| `__init__(port, baud, timeout, ...)` | opens the port with `serial.Serial(...)`, stores the opcodes, creates a `threading.Lock` |
| `_build_packet(cmd, payload)` | `FE FE LEN CMD payload FA` with `LEN = 1 + len(payload) + 1` |
| `_scan_frame(...)` | **resynchronising parser**: skip bytes until `FE FE`, read LEN, read LEN bytes, check tail `FA` and command |
| `read_angles_deg()` | sends `FE FE 02 20 FA`, waits for a `0x20` answer with 12 payload bytes, decodes 6 × int16 / 100 |
| `send_angles_deg(angles, speed)` | packs 6 × (deg × 100) as int16 + a speed byte into `FE FE 0F 22 … FA` |
| `stop_motion()`, `resume_motion()` | `FE FE 02 29 FA` / `FE FE 02 28 FA` |
| `close()` | releases the port |

`with self._lock:` makes each call atomic, which is why the template warns not to run serial cells in parallel.

**How it works**: `os.path.join(a, b)` builds a path; `os.path.isfile(p)` is `True` if that file exists.

### 2. Define the serial connection parameters

The myCobot 280 communicates over a UART serial connection. Define the expected parameters from the codebase.

In [3]:
serial_port = '/dev/serial0'   # Set the serial device used by the robot (Pi UART; link to /dev/ttyAMA0)
baud_rate = 1_000_000          # Set the communication baud rate
timeout_s = 0.05               # Set an appropriate serial timeout (s, per byte read)

print(f'Port: {serial_port}, Baud: {baud_rate}, Timeout: {timeout_s} s')

Port: /dev/serial0, Baud: 1000000, Timeout: 0.05 s


**Concept.** A UART link needs the same speed on both ends. 1 000 000 baud with 10 bits per byte on the wire (8N1) ≈ 100 000
bytes/s: an 18-byte frame takes 0.18 ms, far faster than the arm moves.

**How it works**
- `/dev/serial0` is a **symbolic link** to the Pi's UART (`/dev/ttyAMA0` on this Pi; Ubuntu needed a udev rule for the link,
  README setup step 2). Using the link keeps the notebook independent of that detail.
- `1_000_000`: underscores are allowed as digit separators in Python number literals.
- The timeout is per byte read: short, so a missing answer cannot block the notebook; `read_angles_deg(timeout_s=0.5)`
  retries until its own deadline.

> ### 🧪 Simulation (optional, only without the robot)
> The code cell above is the real-robot version. To run Part 1 **without** the robot:
> 1. Terminal (laptop): `python3 solutions/sim_robot.py` → it prints the virtual port, e.g. `/dev/pts/5`, and then logs
>    every command the simulated arm receives. It starts at the pose read on the real robot in Lab 7 (parked).
> 2. Run the cell below **instead of** the one above (same names, only the port differs), then continue with Task 3.
>    Every Part 1 cell then talks through the real `serial_iface.py` to the simulated arm.
>
> The simulated arm is a model: its speed law (1.5°/s per speed unit) and its behaviour after *stop* (ignore motion until
> *resume*) are assumptions; compare them with the real arm.

In [ ]:
# SIMULATION ONLY: the virtual port printed by solutions/sim_robot.py (not /dev/serial0)
serial_port = '/dev/pts/5'
baud_rate = 1_000_000
timeout_s = 0.05

## Communication Frames and Packet Structure

The myCobot 280 uses structured frames so the controller can identify command boundaries, interpret payloads, and detect invalid data. The frame format is:

```
FE FE LEN CMD [payload] FA
```

- `FE FE` — header (2 bytes)
- `LEN` — byte count from CMD through FA inclusive: `LEN = 1 + len(payload) + 1`
- `CMD` — command type byte
- `[payload]` — command-specific data (may be empty)
- `FA` — tail byte

### 3. Define the protocol constants

Define the header, tail, and key command bytes from the codebase.

In [4]:
HEAD = 0xFE
TAIL = 0xFA

CMD_READ_ANGLES = 0x20
CMD_SEND_ANGLES = 0x22
CMD_RESUME = 0x28
CMD_STOP = 0x29

print(f'Header: 0x{HEAD:02X}, Tail: 0x{TAIL:02X}')
print(f'Read angles: 0x{CMD_READ_ANGLES:02X}')
print(f'Send angles: 0x{CMD_SEND_ANGLES:02X}')
print(f'Stop: 0x{CMD_STOP:02X}, Resume: 0x{CMD_RESUME:02X}')

Header: 0xFE, Tail: 0xFA
Read angles: 0x20
Send angles: 0x22
Stop: 0x29, Resume: 0x28


**Concept.** `FE FE` marks the start (two bytes, so one stray `FE` is not mistaken for a start); `LEN` says how many
bytes follow; `CMD` says what the frame means; the tail `FA` lets the receiver check the frame is complete.

**How it works**: `0x20` is a **hexadecimal** literal (= 32); `f'{x:02X}'` prints an integer as at least two upper-case hex
digits (`0xFE`). Names in capitals are a convention for constants.

### 4. Manually assemble a frame

Build the **stop** frame by hand. A stop command has no payload, so `LEN = 1 (CMD) + 1 (FA) = 2`. The full frame is: `FE FE 02 29 FA`.

Also build the **read-angles** request frame: `FE FE 02 20 FA`.

In [5]:
# Stop frame
stop_frame = bytes([HEAD, HEAD, 0x02, CMD_STOP, TAIL])  # Build the protocol frame for the stop command
print('Stop frame:', stop_frame.hex(' '))

# Read-angles request frame
read_frame = bytes([HEAD, HEAD, 0x02, CMD_READ_ANGLES, TAIL])  # Build the protocol frame for requesting joint angles
print('Read frame:', read_frame.hex(' '))

Stop frame: fe fe 02 29 fa
Read frame: fe fe 02 20 fa


**How it works**
- `bytes([..])` turns a list of integers 0…255 into a `bytes` object, what `serial.write` needs; `.hex(' ')` prints it in hex
  with spaces.
- **LEN rule:** `LEN = 1 (CMD) + len(payload) + 1 (tail)`; stop and read-angles have no payload, so LEN = 2.

**Example: the class builds the same bytes**

```python
from serial_iface import MyCobotSerialInterface
print(MyCobotSerialInterface._build_packet(CMD_STOP).hex(' '), ' == stop_frame:',
      MyCobotSerialInterface._build_packet(CMD_STOP) == stop_frame)
print(MyCobotSerialInterface._build_packet(CMD_SEND_ANGLES, bytes(13)).hex(' '))   # 13 payload bytes -> LEN 0x0F
```

Output:
```text
fe fe 02 29 fa  == stop_frame: True
fe fe 0f 22 00 00 00 00 00 00 00 00 00 00 00 00 00 fa
```

`_build_packet` is a `staticmethod`: callable on the class itself, no object and no open port needed. `bytes(13)` = 13 zero
bytes; with them LEN = 15 = 0x0F, the LEN of every send-angles frame (12 angle bytes + speed).

### 5. Write a frame-printing helper

Create a short function that prints any byte sequence as formatted hex for debugging. This is useful throughout the lab when inspecting raw serial traffic.

In [6]:
def print_frame(label, frame_bytes):
    hex_str = ' '.join(f'{b:02X}' for b in frame_bytes)
    print(f'{label}: [{hex_str}] ({len(frame_bytes)} bytes)')

# Test with the stop frame
print_frame('Stop', stop_frame)
print_frame('Read', read_frame)

Stop: [FE FE 02 29 FA] (5 bytes)
Read: [FE FE 02 20 FA] (5 bytes)


**How it works**: iterating over a `bytes` object yields integers; `' '.join(f'{b:02X}' for b in frame_bytes)` formats each
as two hex digits (a **generator expression** inside `join`) and joins them with spaces.

**Example: how an angle becomes two bytes (int16, 0.01 deg)**

```python
angle = -99.31                                  # deg, joint 2 of the parked pose
v = int(round(angle * 100))                     # -9931 in 0.01-deg units
raw = v + (1 << 16) if v < 0 else v             # two's complement for negative numbers: 65536 - 9931
hi, lo = (raw >> 8) & 0xFF, raw & 0xFF          # high byte first (big-endian)
print('bytes:', f'{hi:02X} {lo:02X}')
back = (hi << 8) | lo
if back & 0x8000:                               # sign bit set -> negative
    back -= 0x10000
print('decoded:', back / 100.0)
import struct
print('struct:', struct.pack('>h', v).hex(' '), struct.unpack('>h', bytes([hi, lo]))[0] / 100.0)
```

Output:
```text
bytes: D9 35
decoded: -99.31
struct: d9 35 -99.31
```

Range −327.68…+327.67°, resolution 0.01°. `<<` / `>>` shift bits (×256 / ÷256), `&` keeps bits, `|` merges them.
`struct.pack('>h', v)` does the same in one call (`>` big-endian, `h` signed 16 bit). This is exactly what
`send_angles_deg` and `read_angles_deg` do.

## Reading Robot State

### 6. Open the serial connection

Instantiate the `MyCobotSerialInterface` from the codebase wrapper. This opens the serial port — make sure the ROS 2 controller is NOT running.

**Note:** The serial interface uses an internal threading lock. Do not run multiple serial cells simultaneously.

**[MOVES] from here to Task 15 (real robot): the ROS controller must not be running** (`fuser -v /dev/serial0` shows who holds the port).

In [ ]:
from serial_iface import MyCobotSerialInterface

ser = MyCobotSerialInterface(
    port=serial_port,
    baud=baud_rate,
    timeout=timeout_s,
    num_joints=6,
    stop_opcode=CMD_STOP,
    resume_opcode=CMD_RESUME,
)
print('Serial port opened.')

**How it works**
- Keyword arguments name each parameter; unnamed ones keep the defaults of `__init__` (e.g. `move_opcode=0x34`).
- The port opens only if no other process holds it; otherwise a "device busy" error: stop the controller first.

### 7. Read the current joint angles

Call `read_angles_deg()` to read the measured joint state. The returned values are in **degrees** (the hardware encoding divides int16 values by 100). These are measured encoder values, not commanded targets.

In [ ]:
measured_deg = ser.read_angles_deg(timeout_s=0.5)  # Read the current joint angles from the robot

if measured_deg is not None:
    print('Measured joint angles (deg):', measured_deg)
else:
    print('No response from robot. Check connection and power.')

**Concept.** These are **measured** encoder values, not the last commanded target.

**How it works**: the method sends `FE FE 02 20 FA`, scans the incoming bytes for a valid `0x20` answer until `timeout_s`, and
returns six floats (deg) or `None`. Always test for `None`: robot off, wrong port or wrong baud must not crash later code.
The parked pose read in the lab: `[7.2, -99.31, -70.92, 81.38, 90.08, -74.26]`.

### 8. Convert and compare with a reference

Convert the measured angles to a NumPy array and compare with the expected home position (all zeros). Note: the myCobot 280 rarely reads exactly zero due to encoder resolution and mechanical play.

In [ ]:
if measured_deg is not None:
    measured_arr = np.array(measured_deg)
    home_deg = np.zeros(6)
    diff_deg = measured_arr - home_deg  # Compute the difference between the measured and home configurations

    print('Measured (deg):', measured_arr)
    print('Home (deg):    ', home_deg)
    print('Difference (deg):', diff_deg)

**Concept.** The parked arm is far from the all-zero pose (differences up to ~99°): zero means "arm straight up".
⚠ So the zeros are a **reference**, never a target from the parked pose: that would be a 99° move at once.

**How it works**: NumPy arrays subtract element by element; a plain list cannot (`[1, 2] - [1, 1]` is a `TypeError`).

## Sending Position Commands

### 9. Define a safe position command

Define a small, safe joint-angle target in **degrees** and a speed level (1–100). Start with a very small motion from the current position.

**Caution:** Ensure the workspace is clear and the motion direction is understood before sending any command.

In [ ]:
# Small safe target — move joint 1 by ~10 degrees from current position
target_deg = (np.array(measured_deg) + np.array([10, 0, 0, 0, 0, 0])).tolist()  # relative to the MEASURED pose
speed = 20       # slow: 20 of 100

limits = np.array([168, 135, 150, 145, 165, 180])           # added: myCobot 280 joint limits (deg), data sheet
assert np.all(np.abs(target_deg) <= limits), 'target outside the joint limits: not sent'

print('Target (deg):', target_deg)
print('Speed:', speed)

**Concept.** A target defined **relative to where the arm is** cannot cause a big jump. (The template's earlier version used
`[90, 0, 0, 90, 0, 0]` at speed 100: up to ~170° per joint at full speed.)

**How it works**
- `np.array(a) + np.array([10, 0, ...])` adds 10° to joint 1 only; `.tolist()` turns the result back into a list.
- `speed` is a percentage of the maximum joint speed (1 = slowest, 100 = fastest).
- `np.abs(target) <= limits` compares all six joints at once; `assert` stops the cell with the message if any fails.

### 10. Send the position command

Send the target using the codebase wrapper. Start with small, well-understood motions and confirm the result before attempting larger movements.

In [ ]:
ser.send_angles_deg(target_deg, speed)
print('Position command sent.')

# Wait for the motion to complete
time.sleep(3.0)

**How it works**: `send_angles_deg` packs `FE FE 0F 22 [6 × int16] [speed] FA` (18 bytes) and writes it. There is **no
answer**: the robot just starts moving, which is why the next task reads the state back. A 10° move at speed 20 takes well
under 3 s.

**Example: the 18 bytes of a send-angles frame**

```python
angles = [17.2, -99.31, -70.92, 81.38, 90.08, -74.26]       # the parked pose with joint 1 + 10 deg
frame = [HEAD, HEAD, 0x0F, CMD_SEND_ANGLES]
for a in angles:
    v = int(round(a * 100)) & 0xFFFF                         # & 0xFFFF = two's complement in 16 bits
    frame += [v >> 8, v & 0xFF]
frame += [20, TAIL]                                          # speed byte, tail
print_frame('send_angles', bytes(frame))
```

Output:
```text
send_angles: [FE FE 0F 22 06 B8 D9 35 E4 4C 1F CA 23 30 E2 FE 14 FA] (18 bytes)
```

`list += [...]` appends several items. `v & 0xFFFF` is a shorter way to get the two's complement of a negative number.
Byte layout: `FE FE` · `0F` (LEN 15) · `22` · 12 angle bytes · speed `14` (= 20) · `FA`.

### 11. Validate: compare commanded and measured angles

Read the robot state again after the motion and compute the difference between the commanded target and the measured position. Small residual differences are normal.

In [ ]:
measured_after = ser.read_angles_deg(timeout_s=0.5)

if measured_after is not None:
    measured_after_arr = np.array(measured_after)
    target_arr = np.array(target_deg)
    error_deg = measured_after_arr - target_arr  # measured minus commanded (positive = further than requested)

    print('Commanded (deg):', target_arr)
    print('Measured (deg): ', measured_after_arr)
    print('Error (deg):    ', error_deg)
    print('Max |error| (deg):', np.max(np.abs(error_deg)))

**Concept.** A command is a request; the measurement is the evidence. On the real arm expect errors of a few hundredths up to
a few tenths of a degree (encoder resolution 0.01°, servo dead band, gravity load): small compared with the 10° move.
A large error means the move is not finished, was rejected (limits, robot in *stopped* state) or misinterpreted.

**How it works**: `np.max(np.abs(e))` = the worst joint.

### 12. Plot commanded vs measured joint values

Visualise the commanded and measured values side by side for all six joints.

In [ ]:
if measured_after is not None:
    joints = np.arange(1, 7)
    fig, ax = plt.subplots(figsize=(8, 4))
    width = 0.3
    ax.bar(joints - width/2, target_arr, width, label='Commanded')
    ax.bar(joints + width/2, measured_after_arr, width, label='Measured')
    ax.set_xlabel('Joint')
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Commanded vs Measured Joint Angles')
    ax.legend()
    ax.grid(True, axis='y')
    plt.tight_layout()

**How it works**: a grouped bar chart; shifting the two series by ±`width/2` puts the bars side by side. The bars will look
identical (errors < 1° next to angles of tens of degrees); to *see* the error, plot `error_deg` instead.

## Stop and Resume Handling

### 13. Send a stop command

The stop command immediately halts robot motion. Use the codebase wrapper method — this is not optional but an essential part of safe robot operation.

In [ ]:
ser.stop_motion()
print('Stop command sent.')

**Concept.** `FE FE 02 29 FA`: the shortest possible frame, quick to send, and it can only make the robot safer. After it the
robot **ignores motion commands until resume** (template text; in the lab on 2026-09-28 the arm once ignored commands
until a restart, probably this state).

### 14. Send a resume command

After a stop, the robot ignores motion commands until a resume is issued. Send the resume command and confirm that the robot accepts commands again.

In [ ]:
ser.resume_motion()
print('Resume command sent.')

# Verify by reading angles
check = ser.read_angles_deg(timeout_s=0.5)
if check is not None:
    print('Robot responding. Angles (deg):', check)
else:
    print('No response after resume.')

**How it works**: `FE FE 02 28 FA`; reading the angles afterwards proves the link still works. A stop without a resume
leaves the robot ignoring commands: a common cause of "the robot doesn't move" later.

### 15. Close the serial connection

Before switching to the ROS 2 controller in Part 2, close the serial port. The controller needs exclusive access to the serial device.

**Reflection:** Why should stop functionality be treated as part of the normal control workflow, not as an exceptional feature? Consider what happens if you issue a command with an unintended target or if the robot moves in an unexpected direction.

In [ ]:
ser.close()
print('Serial port closed. You can now launch the ROS 2 controller.')

**How it works**: only one process can own the port; while the notebook holds it, the controller cannot open it. After a
crash, restart the kernel or find the holder with `fuser -v /dev/serial0`.

**Reflection: why is stop part of the normal workflow?** Any command can turn out wrong: a typo, a sign error, radians where
degrees were expected, a person reaching in. The robot does not check whether a well-formed command is *sensible*. So the
stop path is planned **before** the first move (who, which cell or switch), tried in tests, and short (one frame). A stop
that was never tried is not a safety mechanism.

## Part 2: ROS 2 Controller Integration

In this part you use the ROS 2 controller node (`mycobot_control`) which handles serial communication internally. Launch the controller in a separate terminal before proceeding:

```bash
source ~/venvs/mycobot/bin/activate
ros2 launch mycobot_control mycobot_control.launch.py
```

**Unit convention change:** The direct serial interface uses **degrees** (hardware encoding). The ROS 2 topics use **radians** (ROS convention). The controller converts internally:
- `/mycobot/joint_command`: expects **radians** → controller does `math.degrees(x)`
- `/mycobot/joint_velocity`: expects **rad/s** → mapped to speed 0–100 internally
- `/mycobot/joint_states`: publishes positions in **radians**
- `/mycobot/ee_pose`: expects `[x_mm, y_mm, z_mm, roll_deg, pitch_deg, yaw_deg]`

## Position Mode and Velocity Mode

### 16. Position mode vs velocity mode

**Position mode** (`/mycobot/joint_command`): you specify a target configuration, and the controller drives the robot there. Appropriate when the desired end state is known.

**Velocity mode** (`/mycobot/joint_velocity`): you specify a motion rate, and the robot moves as long as the velocity command is active. Appropriate for continuous or incremental motion. Requires extra care:
- Deadband: velocities below 0.01 rad/s per joint are treated as zero
- Max speed: ~2.618 rad/s (150 deg/s)
- Zero velocity vector triggers an immediate stop
- Acceleration ramp: limited to 1.0 rad/s²

### 17. Inspect the ROS 2 topics

Use the ROS 2 CLI to verify the controller is running and inspect the available topics.

In [7]:
# Run these in a terminal or uncomment to run from the notebook:
# !ros2 topic list
# !ros2 topic info /mycobot/joint_states
# !ros2 topic info /mycobot/joint_command
# !ros2 topic info /mycobot/joint_velocity
# !ros2 topic echo /mycobot/joint_states --once

**Concept (Task 16).** **Position mode** (`/mycobot/joint_command`): a target configuration; for each message the
controller sends one `send_angles` at speed 50 and the robot drives there itself. **Velocity mode**
(`/mycobot/joint_velocity`): a rate; the controller integrates it into angle targets at 20 Hz and streams them while the
command stays active (deadband 0.01 rad/s, max ≈ 2.6 rad/s, all-zeros = stop, ramp 1 rad/s²).

**How it works (Task 17)**: `!command` runs a shell command from a notebook cell. Expected: `/mycobot/joint_states` is
`sensor_msgs/msg/JointState` with 1 publisher (the controller); `/mycobot/joint_command` and `/mycobot/joint_velocity` are
`std_msgs/msg/Float64MultiArray` with 1 subscriber each.

## ROS 2 Node Integration

### 18. Create a notebook node

Create a small ROS 2 node that subscribes to `/mycobot/joint_states` and stores the most recent measured state.

In [8]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float64MultiArray


class SerialLabNode(Node):
    def __init__(self):
        super().__init__('serial_lab_node')
        self.last_js = None
        self.js_sub = self.create_subscription(
            JointState, '/mycobot/joint_states', self._js_cb, 1,    # 1 (was 10): keep only the newest state
        )

    def _js_cb(self, msg):
        self.last_js = msg

**How it works**: the Lab 2 pattern: a `Node` subclass whose subscription callback stores the latest message.
`self.last_js = None` is the "nothing received yet" marker. The callback only runs while the node is spun.
The last argument of `create_subscription` is the **queue depth**: how many unprocessed messages are kept. With 10, a
`spin_once` after a few seconds of waiting hands you the *oldest* of the last ten states (2 s old at 5 Hz); with 1 it is
always the newest, which is what "read the current state" means.

> ### 🧪 Simulation (optional, only without the robot)
> To run Part 2 **without** the robot: keep `sim_robot.py` from Part 1 running (Task 15 closed the notebook's port),
> then start the **real** controller node on that port, isolated from the lab network:
> ```bash
> source /opt/ros/jazzy/setup.bash && source ros2_ws/install/setup.bash
> export ROS_DOMAIN_ID=99 ROS_AUTOMATIC_DISCOVERY_RANGE=LOCALHOST
> ros2 run mycobot_control mycobot_control --ros-args -p port:=/dev/pts/5 -p read_timeout_s:=0.75
> ```
> Then run the cell below **before Task 19** (it must come before `rclpy.init()`).
>
> ⚠ Without the isolation, a simulation on a lab computer would also command the real controller on the Pi.

In [ ]:
# SIMULATION ONLY: run before rclpy.init() (after a kernel restart), never in the lab with the real robot
import os
os.environ['ROS_DOMAIN_ID'] = '99'                           # not the lab's domain id
os.environ['ROS_AUTOMATIC_DISCOVERY_RANGE'] = 'LOCALHOST'    # talk only to nodes on this computer

### 19. Initialize and spin

Initialize `rclpy`, create the node, and spin once to receive the latest joint state.

**[MOVES] from Task 21 on (real robot): `mycobot_control` running (`ros2 launch mycobot_control mycobot_control.launch.py`), workspace clear.**

In [ ]:
if not rclpy.ok():
    rclpy.init(args=None)

node = SerialLabNode()
for _ in range(50):                       # added: spin until a state arrives (one spin_once missed it in the lab)
    rclpy.spin_once(node, timeout_sec=0.1)
    if node.last_js is not None:
        break

if node.last_js is not None:
    print('Joint names:', node.last_js.name)
    print('Positions (rad):', list(node.last_js.position))
    print('Positions (deg):', [np.degrees(p) for p in node.last_js.position])
else:
    print('No joint state received. Is the controller running?')

**How it works**: `rclpy.init` once per process; `spin_once(node, timeout_sec)` handles at most one waiting event. Right
after creating the node, discovery can take a moment, so one call can miss the first message (it did in the lab): the
loop retries for up to 5 s. Degrees should match Part 1's serial reading.

### 20. Create a position-command publisher

Create a publisher for `/mycobot/joint_command`. This topic expects joint angles in **radians**.

In [ ]:
pos_pub = node.create_publisher(Float64MultiArray, '/mycobot/joint_command', 10)

**How it works**: `create_publisher(type, topic, queue_size)`; the topic expects **radians**.

### 21. Publish a safe position command (radians)

Send a small safe joint-position command in **radians** via the ROS 2 position-mode path. This is the higher-level equivalent of the direct `send_angles_deg()` used in Part 1.

In [ ]:
q_cmd_rad = np.array(node.last_js.position) + np.radians([10, 0, 0, 0, 0, 0])  # joint 1 + 10 deg from the measured pose

msg = Float64MultiArray()
msg.data = q_cmd_rad.tolist()  # the message needs a plain list of floats
pos_pub.publish(msg)
print(f'Published position command (rad): {q_cmd_rad}')

# Wait and read back
time.sleep(3.0)
pos_pub.publish(msg)                      # added: same target again, no motion, but the controller re-reads the arm
time.sleep(0.5)                           # added: let the refreshed state arrive
rclpy.spin_once(node, timeout_sec=0.5)
if node.last_js is not None:
    print('Measured (rad):', list(node.last_js.position))

**Concept.** The ROS twin of Task 10: the controller converts radians → degrees and calls `send_angles` (speed 50).

**How it works**
- `np.radians([10, 0, ...])` = 0.1745 rad on joint 1; `.tolist()` because `Float64MultiArray.data` must be a Python
  sequence.
- The added re-publish exists because of how the controller reports the state (note below): the same target again after
  the move costs no motion but makes the controller read the arm, so the read-back shows the new pose and the next
  velocity command starts from the real pose (without it, the simulation showed joint 1 driven back by the stale 10°).

⚠ **What `/mycobot/joint_states` really is** (read in `mycobot_control.py`, `_state_timer`; seen in the simulation,
not yet checked on the robot): the controller publishes its **estimate** of the angles at 5 Hz and reads the robot only
right after a command. After a `/mycobot/joint_command` that read happens immediately, before the arm has moved, so the
published pose stays at the *old* angles until the next command. Two consequences: (1) the "measured" pose you read
can lag one move behind; (2) a **velocity** command sent after a position command first drives the joints back towards
that stale estimate. Refresh the estimate after a position move by publishing the same target again once the arm has
stopped (no motion, but a new read). (3) Ending a velocity command with zeros makes the controller send **STOP** to the
robot, and the position path (`_pos_cb`) never sends **RESUME**; if the robot ignores motion after STOP (as the Lab 7
template says, and the simulation does), every later `/mycobot/joint_command` is ignored until a new velocity command
makes the controller resume. Wake-up: publish one small velocity message (e.g. 0.02 rad/s on one joint), then the position
command. **[check] on the robot.**

### 22. Create a velocity-command publisher

Create a publisher for `/mycobot/joint_velocity`. This topic expects joint velocities in **rad/s**.

In [ ]:
vel_pub = node.create_publisher(Float64MultiArray, '/mycobot/joint_velocity', 10)

**How it works**: same as Task 20, for rates in **rad/s**.

### 23. Publish a safe velocity command

Send a very small velocity on one joint for a short duration, then send zeros to stop.

**Safety warning:**
- Keep velocities small (e.g., 0.1 rad/s on one joint)
- Keep duration short (e.g., 1–2 seconds)
- Be ready to send zeros or use the stop mechanism immediately
- Velocities below 0.01 rad/s are ignored (deadband)
- Sending `[0, 0, 0, 0, 0, 0]` triggers an immediate stop

In [ ]:
# Small safe velocity: rotate joint 1 slowly
qdot_cmd = np.array([0.1, 0.0, 0.0, 0.0, 0.0, 0.0])  # rad/s, joint 1 only
duration = 1.5  # s  -> about 0.15 rad = 8.6 deg

rate_hz = 20.0
msg = Float64MultiArray()

start = time.time()
try:                                      # added: the stop below is sent even after an error or an interrupt
    while time.time() - start < duration:
        msg.data = qdot_cmd.tolist()  # the message needs a plain list of floats
        vel_pub.publish(msg)
        rclpy.spin_once(node, timeout_sec=0.0)
        time.sleep(1.0 / rate_hz)
finally:
    # Stop: send zero velocity
    msg.data = [0.0] * 6
    vel_pub.publish(msg)
print('Velocity command completed. Stop sent.')

**Concept.** Velocity mode keeps moving as long as messages arrive; the all-zero message is the stop.

**How it works**
- The `while` loop publishes at 20 Hz for 1.5 s; `spin_once(node, 0.0)` processes incoming states without waiting.
- `try: ... finally:` runs the `finally` block **always** (normal end, exception, or "interrupt kernel").
- The controller ramps the rate up at 1 rad/s², so the joint needs ~0.1 s to reach 0.1 rad/s and moves slightly less than
  0.15 rad: an example for Task 26.

## Commanded vs Measured Motion

### 24. Record measured joint states over time

Send a position command and record the measured `/mycobot/joint_states` at regular intervals to observe the motion dynamics.

In [ ]:
# Send a target
rclpy.spin_once(node, timeout_sec=0.5)                     # added: newest state (queue depth 1, Task 18)
q_target_rad = np.array(node.last_js.position) - np.radians([10, 0, 0, 0, 0, 0])  # joint 1 back by 10 deg
vel_pub.publish(Float64MultiArray(data=[0.02, 0.0, 0.0, 0.0, 0.0, 0.0]))   # added: wake-up after Task 23's STOP (note)
time.sleep(0.1)
msg = Float64MultiArray()
msg.data = q_target_rad.tolist()
pos_pub.publish(msg)

# Record measured states
timestamps = []
measured_positions = []
start = time.time()
record_duration = 4.0

while time.time() - start < record_duration:
    rclpy.spin_once(node, timeout_sec=0.05)
    if node.last_js is not None:
        timestamps.append(time.time() - start)
        measured_positions.append(list(node.last_js.position))
    time.sleep(0.1)

timestamps = np.array(timestamps)
measured_positions = np.array(measured_positions)
print(f'Recorded {len(timestamps)} samples over {record_duration} s')

**How it works**
- The **wake-up** line: Task 23 ended with zeros, so the controller sent STOP to the robot, and a position command alone
  does not resume it (note under Task 21, point 3). One velocity message of 0.02 rad/s (above the 0.01 deadband) makes the
  controller send RESUME; the position command right after cancels that tiny stream. In the simulation, without this line
  the command was ignored (`send_angles ignored (stopped)`). [check] on the robot: try Task 24 once without it.
- `Float64MultiArray(data=[...])` sets the field in the constructor (same as `msg.data = ...`).
- Two lists collect time and the six joint values; converted to arrays afterwards (`measured_positions` has shape
  samples × 6). About 10 samples per second; the controller publishes at 5 Hz, so neighbouring samples repeat.

### 25. Plot the motion over time

Visualise how each joint moves toward the commanded target over time.

In [ ]:
if len(timestamps) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    for j in range(measured_positions.shape[1]):
        ax.plot(timestamps, np.degrees(measured_positions[:, j]), label=f'q{j+1}')
        ax.axhline(np.degrees(q_target_rad[j]), color=f'C{j}', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Measured joint motion toward commanded target')
    ax.legend(fontsize=7)
    ax.grid(True)
    plt.tight_layout()

**How it works**: one solid line per joint (recorded), one dashed line per joint at the target in the same colour
(`color=f'C{j}'` = the *j*-th default colour).

**What you will probably see** (simulation; [check] on the robot): joint 1's first sample **on** the target, then the
line back at the **old** angle, flat for the whole 4 s, 10° away from the dashed target, although the arm did move. That is
not the arm, it is how `mycobot_control` reports the state (note under Task 21): for a `joint_command` it publishes the
commanded angles once as its estimate, then reads the arm immediately (before it has moved) and republishes that old value
until the next command. Task 26 uses this.

### 26. Interpret commanded vs measured differences

Why might commanded and measured values differ? Consider:
- Motion still in progress (robot has not yet reached the target)
- Sampling delay (discrete read intervals vs continuous motion)
- Internal controller limits (acceleration ramp, speed cap)
- Mechanical play and encoder resolution

In [9]:
# Space for student interpretation.
# -> answers in the markdown cell below

**Answers**
1. **Motion in progress:** a sample taken during the move lies between start and target; judge the final error at the end.
2. **Sampling delay:** the state comes at 5 Hz and is read at discrete times: a staircase up to 0.2 s late.
3. **Controller limits:** speed cap, acceleration ramp (1 rad/s²), `max_step_deg` (2° per tick when streaming).
4. **Mechanics and encoders:** backlash, servo dead band, gravity load, 0.01° resolution leave a small final error.
5. **What is "measured" here:** `/mycobot/joint_states` is the controller's **estimate**, refreshed from the arm only right
   after a command. After a position command it can show the pre-move angles for as long as no new command arrives,
   which is what Task 25's flat line shows. For a true time history read the arm directly (Part 1: `read_angles_deg()` in
   a loop) or make the controller read again (publish the target again).

## Safety and Practical Precautions

### 27. Pre-motion safety checklist

Before enabling any robot motion, verify:
- Workspace is clear of obstacles and people
- Motion direction and magnitude are understood
- Stop mechanism is known and accessible
- Commands are modest during initial tests
- Serial port or controller is confirmed active

In [10]:
# Students can use this cell to print a confirmation or checklist before running motion commands.
print('Safety checklist confirmed: [ ]')  # Students check this manually

Safety checklist confirmed: [ ]


**How to use it**: a person ticks each item *in the room*, not in code: workspace clear, direction and size of the move
understood, stop (power switch) within reach, small and slow first, the right program owns the port (serial **or**
controller). The cell just makes you say it before each motion session.

### 28. Why real-robot operation requires extra caution

Briefly explain why direct communication with a real robot requires more caution than simulated motion. Consider physical consequences, irreversible actions, and the gap between model assumptions and hardware behaviour.

In [11]:
# Space for student reflection.
# -> answer in the markdown cell below

**Answer.** A real arm acts on the first well-formed command: no undo, no "are you sure". It can injure people and break
itself, the tool or the table; a collision cannot be reset like a simulation. The model differs from the hardware
(friction, backlash, gravity, latency, a slightly different geometry than the URDF), the robot may be in an unexpected
pose or a stopped state, and communication can fail silently. Hence: validate targets, small steps first, read the state
back, keep the stop ready, never run unattended.

## Reflection

### 29. Summary reflection

Summarise briefly:
1. What does the serial protocol contribute to reliable robot interaction?
2. Why does the frame structure (`FE FE LEN CMD ... FA`) matter?
3. Why is comparing commanded vs measured motion essential?
4. When is position mode appropriate vs velocity mode?

In [12]:
# Space for student answers.
# -> answers in the markdown cell below

# Optional cleanup:
# node.destroy_node()
# rclpy.shutdown()

**Answers**
1. **What the protocol contributes:** a byte-exact language: what each command means, how numbers are encoded (int16 ×
   0.01°), where messages start and end, how state is read back. It turns a byte stream into checkable messages.
2. **Why the frame structure matters:** `FE FE` lets the receiver find a start and **resynchronise** after garbage; `LEN`
   says how many bytes to read; the tail and the length check reveal truncated or corrupted frames; `CMD` selects the
   meaning. Without framing one lost byte would shift every later value.
3. **Why compare commanded and measured:** the command is a request, the measurement the evidence: it shows whether the
   link works, the command was accepted, how fast and how accurately the arm follows, and reveals stopped states, limits
   and faults.
4. **Position vs velocity mode:** position when the goal configuration is known and the path between does not matter (one
   command); velocity for continuous or incremental motion (jogging, Jacobian control, Lab 4), which needs small rates, a
   supervisor and an immediate stop.